<a href="https://colab.research.google.com/github/shafinnahian/teen-mental-health-predictive-model/blob/main/notebooks/digital_wellbeing_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Digital wellbeing classifier

Run this notebook top to bottom on a fresh Colab runtime.

We predict `digital_wellbeing_flag` (Healthy, Moderate, At Risk) from nine lifestyle variables. Stress, anxiety, addiction, and derived risk scores are excluded from the feature set.

After cloning the repo, the dataset is at:

`/content/teen-mental-health-predictive-model/data/Teen_Mental_Health.csv`

Dataset: [Kaggle — Teen Mental Health (argonnxx)](https://www.kaggle.com/datasets/argonnxx/teen-mental-health)

**Part 1 — Setup:** paths, dependencies, load CSV, verify locked config.

**Part 2 — EDA:** leakage check, class balance, lifestyle figures saved to `outputs/figures/`.

**Part 3 — Preprocessing:** stratified split, encoding, scaling, one engineered feature, saved artifacts.

**Part 4 — Modeling:** multinomial logistic regression + random forest, train-only CV tuning, saved pipelines.

**Part 5 — Evaluation:** baselines, train/test metrics, confusion matrix, `outputs/metrics/report.json`.

**Part 6 — Run summary:** saves key results to `outputs/run_summary.zip` for download.

Course project only. Not for clinical use.


In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/teen-mental-health-predictive-model")
DATA_CSV = REPO / "data" / "Teen_Mental_Health.csv"

if not REPO.is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/shafinnahian/teen-mental-health-predictive-model.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

print("Repo:", REPO)
print("Data CSV:", DATA_CSV)
assert DATA_CSV.is_file(), f"Missing dataset at {DATA_CSV}"


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Optional

REPO = Path("/content/teen-mental-health-predictive-model")
DATA_CSV = REPO / "data" / "Teen_Mental_Health.csv"

if REPO.is_dir():
    os.chdir(REPO)


def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src" / "config.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Clone the repo into /content/teen-mental-health-predictive-model first."
    )


PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)
print("Data CSV:", DATA_CSV)

%pip install -q -r {PROJECT_ROOT / "requirements.txt"}


In [ ]:
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import config
import paths
import data_loader
import io_utils

print("Imports OK:", [config.__name__, paths.__name__, data_loader.__name__, io_utils.__name__])
print("src on sys.path:", SRC_PATH)

In [ ]:
from paths import (
    ensure_output_dirs,
    data_csv_path,
    figures_dir,
    models_dir,
    metrics_dir,
    get_project_root,
)

ensure_output_dirs()
print("PROJECT_ROOT (paths.py):", get_project_root())
print("data CSV:", data_csv_path())
print("figures:", figures_dir())
print("models:", models_dir())
print("metrics:", metrics_dir())

In [ ]:
from data_loader import load_raw_data, validate_data

df = load_raw_data()
validation = validate_data(df)
display(df.head())
validation

In [ ]:
from config import (
    TARGET,
    TASK,
    TARGET_CLASSES,
    LIFESTYLE_FEATURES,
    EXCLUDED_COLUMNS,
    SEED,
    TEST_SIZE,
    DATASET_URL,
    EXPECTED_ROWS,
    EXPECTED_COLS,
)

print("TARGET:", TARGET)
print("TASK:", TASK)
print("TARGET_CLASSES:", TARGET_CLASSES)
print("SEED:", SEED)
print("TEST_SIZE:", TEST_SIZE)
print("Expected shape:", (EXPECTED_ROWS, EXPECTED_COLS))
print("Dataset URL:", DATASET_URL)
print("Lifestyle features (" + str(len(LIFESTYLE_FEATURES)) + "):")
for col in LIFESTYLE_FEATURES:
    print(" -", col)
print("Excluded columns:")
for col in EXCLUDED_COLUMNS:
    print(" -", col)

In [ ]:
assert config.TARGET == "digital_wellbeing_flag"
assert df.shape == (1200, 16)
assert validation["missing"] == 0
assert figures_dir().is_dir()
assert models_dir().is_dir()
assert metrics_dir().is_dir()

print("Phase 1 smoke tests passed.")
print("Target counts:", validation["target_counts"])

## Exploratory data analysis

The CSV is already loaded above. This section checks for target leakage, plots class balance and lifestyle patterns, and saves figures to `outputs/figures/`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from config import TARGET, TARGET_CLASSES, EXCLUDED_COLUMNS, SEED

LIFESTYLE_NUMERIC = [
    "age",
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
]

FIGURES_DIR = figures_dir()
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)

print("FIGURES_DIR:", FIGURES_DIR)
print("Target counts:", validation["target_counts"])

## Leakage check

On every row, `mental_health_risk_score` equals `stress_level + anxiety_level + addiction_level`.

Those psychological columns stay out of the model. The table below shows their mean values by wellbeing class for documentation only.


In [ ]:
risk_equals_psycho_sum = (
    df["stress_level"] + df["anxiety_level"] + df["addiction_level"]
    == df["mental_health_risk_score"]
).all()

assert risk_equals_psycho_sum, (
    "Expected mental_health_risk_score == stress + anxiety + addiction for all rows."
)
print(
    "Leakage check PASSED: risk_score = stress + anxiety + addiction (all",
    len(df),
    "rows)",
)
print("Excluded predictors (not used for modeling):", EXCLUDED_COLUMNS)

psycho_by_wellbeing = (
    df.groupby(TARGET)[
        [
            "stress_level",
            "anxiety_level",
            "addiction_level",
            "mental_health_risk_score",
        ]
    ]
    .mean()
    .reindex(list(TARGET_CLASSES))
    .round(2)
)
display(psycho_by_wellbeing)

## Figures

Four plots, saved to `outputs/figures/`. Wellbeing classes are ordered Healthy, Moderate, At Risk.


In [ ]:
# Figure 1 — Class balance
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(
    data=df,
    x=TARGET,
    order=list(TARGET_CLASSES),
    hue=TARGET,
    hue_order=list(TARGET_CLASSES),
    palette="Set2",
    legend=False,
    ax=ax,
)
ax.set_title("Class balance: digital_wellbeing_flag")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")
plt.tight_layout()
out1 = FIGURES_DIR / "01_class_balance.png"
fig.savefig(out1, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out1)

In [ ]:
# Group means for daily social media hours (summary evidence)
social_media_stats = (
    df.groupby(TARGET)["daily_social_media_hours"]
    .agg(["mean", "median", "count"])
    .reindex(list(TARGET_CLASSES))
    .round(2)
)
display(social_media_stats)

In [ ]:
# Figure 2 — Social media hours by wellbeing class (boxplot)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(
    data=df,
    x=TARGET,
    y="daily_social_media_hours",
    order=list(TARGET_CLASSES),
    hue=TARGET,
    hue_order=list(TARGET_CLASSES),
    palette="Set2",
    legend=False,
    ax=ax,
)
ax.set_title("Daily social media hours by digital wellbeing class")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Daily social media hours")
plt.tight_layout()
out2 = FIGURES_DIR / "02_social_media_by_wellbeing.png"
fig.savefig(out2, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out2)

In [ ]:
# Figure 3 — Correlation heatmap (lifestyle numerics only; no psycho scales)
corr = df[LIFESTYLE_NUMERIC].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Correlation heatmap — lifestyle numeric predictors")
plt.tight_layout()
out3 = FIGURES_DIR / "03_lifestyle_correlation.png"
fig.savefig(out3, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out3)
print("Columns in heatmap:", LIFESTYLE_NUMERIC)

In [ ]:
# Figure 4 — Platform usage × wellbeing (row-normalized %)
ct_counts = pd.crosstab(df["platform_usage"], df[TARGET])
ct_counts = ct_counts.reindex(columns=list(TARGET_CLASSES))
ct_pct = ct_counts.div(ct_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(
    ct_pct,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("Platform usage × wellbeing (row % within platform)")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Platform usage")
plt.tight_layout()
out4 = FIGURES_DIR / "04_platform_crosstab.png"
fig.savefig(out4, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out4)
print("Raw counts:")
display(ct_counts)

In [ ]:
# Smoke check: all four figures written
expected_figs = [
    FIGURES_DIR / "01_class_balance.png",
    FIGURES_DIR / "02_social_media_by_wellbeing.png",
    FIGURES_DIR / "03_lifestyle_correlation.png",
    FIGURES_DIR / "04_platform_crosstab.png",
]
for path in expected_figs:
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty figure: {path}"
print("Phase 2 figure smoke tests passed.")
for path in expected_figs:
    print(" -", path)

## Summary

The file has 1,200 rows, 16 columns, and no missing values. Class counts match the locked Kaggle snapshot: Moderate 743, Healthy 306, At Risk 151. Moderate is the majority class, so accuracy alone would be a weak metric; macro-F1 is more informative for later modeling.

`daily_social_media_hours` shows the clearest separation between classes. Healthy teens average about 2.6 hours per day; At Risk averages about 7.1 hours (see the group means table and boxplot). Platform choice shows smaller differences than hours of use.

The lifestyle numeric correlation matrix does not show strong redundancy among the nine predictors at this stage.

Stress, anxiety, addiction, and the derived risk score remain excluded from modeling. Verified on all rows: risk score equals the sum of the three psychological scales.


## Preprocessing

Lifestyle predictors only. We stratify an 80/20 train/test split on `digital_wellbeing_flag`, then fit encoding and scaling on the training rows alone.

Engineered feature (decision, not a validated finding):

`high_screen_before_bed = 1 if screen_time_before_sleep > 2.0 else 0`

Categoricals (`gender`, `platform_usage`, `social_interaction_level`) are one-hot encoded with the first level dropped. Numeric lifestyle columns are standardized. No model training in this section.


In [ ]:
import features
import preprocessing
from config import LIFESTYLE_FEATURES, SEED, TARGET, TARGET_CLASSES, TEST_SIZE, EXCLUDED_COLUMNS
from io_utils import save_joblib
from paths import models_dir

# Fail fast with Colab setup hints if src/ was not cloned or wired correctly.
diag = features.verify_colab_src_setup()
print("Colab/src diagnostics OK")
print("  features loaded from:", diag["module_file"])
print("  src on sys.path:", diag["src_on_sys_path_count"] > 0)
print("preprocessing loaded from:", preprocessing.__file__)
print("models_dir:", models_dir())


In [ ]:
X, y = preprocessing.make_xy(df)

assert list(X.columns) == LIFESTYLE_FEATURES
assert y.name == TARGET
for col in EXCLUDED_COLUMNS:
    assert col not in X.columns, f"Excluded column leaked into X: {col}"

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Lifestyle columns:", list(X.columns))


In [ ]:
X_train, X_test, y_train, y_test = preprocessing.stratified_split(X, y)

train_idx = X_train.index
test_idx = X_test.index

split_counts = pd.DataFrame(
    {
        "train": y_train.value_counts().reindex(list(TARGET_CLASSES)),
        "test": y_test.value_counts().reindex(list(TARGET_CLASSES)),
    }
)
split_pct = pd.DataFrame(
    {
        "train_%": (y_train.value_counts(normalize=True) * 100)
        .reindex(list(TARGET_CLASSES))
        .round(1),
        "test_%": (y_test.value_counts(normalize=True) * 100)
        .reindex(list(TARGET_CLASSES))
        .round(1),
    }
)

print(f"train={len(X_train)} test={len(X_test)} (TEST_SIZE={TEST_SIZE}, SEED={SEED})")
display(split_counts)
display(split_pct)


In [ ]:
pipe = preprocessing.build_preprocessing_pipeline()
print(pipe)


In [ ]:
pipe, X_train_processed, X_test_processed, feature_names = (
    preprocessing.fit_transform_train_test(X_train, X_test, pipe)
)

print("X_train_processed:", getattr(X_train_processed, "shape", None))
print("X_test_processed:", getattr(X_test_processed, "shape", None))
print("n_features:", len(feature_names))
print("feature_names_out:")
for name in feature_names:
    print(" -", name)


In [ ]:
# Smoke checks: lifestyle-only, train-only fit, shapes, stratification
assert X_train_processed.shape == (960, preprocessing.EXPECTED_PROCESSED_N_FEATURES)
assert X_test_processed.shape == (240, preprocessing.EXPECTED_PROCESSED_N_FEATURES)
assert len(feature_names) == preprocessing.EXPECTED_PROCESSED_N_FEATURES

expected_train = {"Moderate": 594, "Healthy": 245, "At Risk": 121}
expected_test = {"Moderate": 149, "Healthy": 61, "At Risk": 30}
for label in TARGET_CLASSES:
    assert int(y_train.value_counts()[label]) == expected_train[label]
    assert int(y_test.value_counts()[label]) == expected_test[label]

# Engineered column present after engineer step (train copy only for inspection)
X_train_eng = features.add_engineered_features(X_train)
assert features.ENGINEERED_FEATURE in X_train_eng.columns
assert set(X_train_eng[features.ENGINEERED_FEATURE].unique()).issubset({0, 1})

print("Phase 3 smoke tests passed.")
print("Preprocessor was fit on train only; test was transformed with that fit.")


In [ ]:
artifact = preprocessing.build_split_artifact(
    train_idx=train_idx,
    test_idx=test_idx,
    X_train_raw=X_train,
    X_test_raw=X_test,
    y_train=y_train,
    y_test=y_test,
    X_train_processed=X_train_processed,
    X_test_processed=X_test_processed,
    feature_names_out=feature_names,
)

pipe_path = models_dir() / preprocessing.PREPROCESS_PIPELINE_FILENAME
split_path = models_dir() / preprocessing.TRAIN_TEST_SPLIT_FILENAME

save_joblib(pipe_path, pipe)
save_joblib(split_path, artifact)

print("Saved:", pipe_path)
print("Saved:", split_path)
assert pipe_path.is_file() and pipe_path.stat().st_size > 0
assert split_path.is_file() and split_path.stat().st_size > 0


## Preprocessing summary

Split: stratified 80/20 on `digital_wellbeing_flag`, `random_state=42` (960 train, 240 test). Class shares match the full data within rounding.

Encoding: one-hot for `gender`, `platform_usage`, and `social_interaction_level` (`drop="first"`). Numeric lifestyle columns use `StandardScaler`. The engineered binary `high_screen_before_bed` is passed through without scaling.

Formula: `high_screen_before_bed = 1 if screen_time_before_sleep > 2.0 else 0`. Threshold is fixed by decision, not learned from labels.

Processed width is 14 columns. Artifacts: `outputs/models/preprocess_pipeline.joblib` and `outputs/models/train_test_split.joblib`.

No classifier was trained here. Modeling is Phase 4.


## Modeling

Two multiclass classifiers on lifestyle predictors only: multinomial logistic regression and a random forest. Each model is a full `Pipeline` (feature engineering + preprocessing + classifier) fit on raw training rows.

Hyperparameters are chosen with stratified 5-fold `GridSearchCV` on the **training set only**, scoring `f1_macro`. The held-out test set is not scored here; that is Phase 5.


In [ ]:
import models
from io_utils import load_joblib, save_joblib, save_json
from paths import models_dir
from preprocessing import PREPROCESS_PIPELINE_FILENAME, TRAIN_TEST_SPLIT_FILENAME

diag = features.verify_colab_src_setup()
print("models loaded from:", models.__file__)
print("models_dir:", models_dir())

pipe_path = models_dir() / PREPROCESS_PIPELINE_FILENAME
split_path = models_dir() / TRAIN_TEST_SPLIT_FILENAME
assert pipe_path.is_file() and pipe_path.stat().st_size > 0, f"Missing {pipe_path}"
assert split_path.is_file() and split_path.stat().st_size > 0, f"Missing {split_path}"
print("Phase 3 artifacts OK:", pipe_path.name, split_path.name)


In [ ]:
# Prefer in-memory Phase 3 splits; fall back to the saved train_test_split artifact.
try:
    X_train_model = X_train
    y_train_model = y_train
    print("Using in-memory X_train / y_train from Phase 3.")
except NameError:
    split_artifact = load_joblib(models_dir() / TRAIN_TEST_SPLIT_FILENAME)
    X_train_model = split_artifact["X_train_raw"]
    y_train_model = split_artifact["y_train"]
    print("Loaded X_train_raw / y_train from train_test_split.joblib.")

assert len(X_train_model) == 960
assert len(y_train_model) == 960
print("Train rows:", len(X_train_model))
print("Train class counts:")
print(y_train_model.value_counts().reindex(list(TARGET_CLASSES)))


### Multinomial logistic regression

For classes $k \in \{\mathrm{Healthy}, \mathrm{Moderate}, \mathrm{At Risk}\}$, multinomial logistic regression models

$$
P(y=k \mid \mathbf{x}) = \mathrm{softmax}(\mathbf{w}_k^\top \mathbf{x} + b_k).
$$

We use `LogisticRegression(solver="lbfgs", max_iter=1000)`. With three classes, scikit-learn minimizes the multinomial (softmax) loss; the deprecated `multi_class` argument is omitted. The single tuned hyperparameter is regularization strength `C` in `{0.1, 1.0, 10.0}`.


In [ ]:
logistic_pipe = models.build_model_pipeline(models.build_logistic_classifier())
logistic_search = models.tune_and_fit_pipeline(
    logistic_pipe,
    models.logistic_param_grid(),
    X_train_model,
    y_train_model,
)
logistic_best = logistic_search.best_estimator_

print("Logistic best_params_:", logistic_search.best_params_)
print(
    "Logistic best_cv_f1_macro (train CV only, not test):",
    round(float(logistic_search.best_score_), 4),
)

logistic_path = models_dir() / models.LOGISTIC_PIPELINE_FILENAME
save_joblib(logistic_path, logistic_best)
print("Saved:", logistic_path)

phase4_meta = {
    "logistic": {
        "best_params": {k: v for k, v in logistic_search.best_params_.items()},
        "best_cv_f1_macro": round(float(logistic_search.best_score_), 6),
        "scoring": models.CV_SCORING,
        "cv_n_splits": models.CV_N_SPLITS,
    }
}


### Random forest

A `RandomForestClassifier` with `n_estimators=100` and `random_state=42` averages many decision trees. Tree depth is the main overfitting control we tune: `max_depth` in `{None, 10, 20}` via the same stratified train-only `GridSearchCV` and `f1_macro` scoring.


In [ ]:
rf_pipe = models.build_model_pipeline(models.build_rf_classifier())
rf_search = models.tune_and_fit_pipeline(
    rf_pipe,
    models.rf_param_grid(),
    X_train_model,
    y_train_model,
)
rf_best = rf_search.best_estimator_

print("RF best_params_:", rf_search.best_params_)
print(
    "RF best_cv_f1_macro (train CV only, not test):",
    round(float(rf_search.best_score_), 4),
)

rf_path = models_dir() / models.RF_PIPELINE_FILENAME
save_joblib(rf_path, rf_best)
print("Saved:", rf_path)

phase4_meta["random_forest"] = {
    "best_params": {k: v for k, v in rf_search.best_params_.items()},
    "best_cv_f1_macro": round(float(rf_search.best_score_), 6),
    "scoring": models.CV_SCORING,
    "cv_n_splits": models.CV_N_SPLITS,
}

meta_path = models_dir() / models.PHASE4_MODEL_META_FILENAME
save_json(meta_path, phase4_meta)
print("Saved:", meta_path)
print(phase4_meta)


In [ ]:
# Smoke tests: reload, predict on a tiny train slice only (no test scoring).
reloaded_logistic = load_joblib(logistic_path)
reloaded_rf = load_joblib(rf_path)

for name, pipe in (("logistic", reloaded_logistic), ("rf", reloaded_rf)):
    step_names = [s[0] for s in pipe.steps]
    assert step_names == ["engineer", "preprocess", "classifier"], (name, step_names)

sample = X_train_model.iloc[:5]
pred_log = reloaded_logistic.predict(sample)
pred_rf = reloaded_rf.predict(sample)
assert len(pred_log) == 5 and len(pred_rf) == 5
assert set(pred_log).issubset(set(TARGET_CLASSES))
assert set(pred_rf).issubset(set(TARGET_CLASSES))

assert logistic_path.is_file() and logistic_path.stat().st_size > 0
assert rf_path.is_file() and rf_path.stat().st_size > 0
assert meta_path.is_file() and meta_path.stat().st_size > 0

print("Phase 4 smoke tests passed.")
print("logistic sample preds:", list(pred_log))
print("rf sample preds:", list(pred_rf))
print("No test-set metrics computed in Phase 4.")


## Modeling summary

Both models are full pipelines (engineer → preprocess → classifier) trained on lifestyle features only.

Hyperparameters were selected with stratified 5-fold `GridSearchCV` on the training set, scoring `f1_macro`. Chosen values are printed above and stored in `outputs/models/phase4_model_meta.json`.

Artifacts: `outputs/models/logistic_pipeline.joblib`, `outputs/models/rf_pipeline.joblib`.

Test-set comparison tables, baselines, and confusion matrices follow in Part 5. Course project only — not for clinical use.


## Evaluation

Phase 5 scores the two saved pipelines and two simple baselines on **train and test**.

Primary metric is macro-$F_1$ (unweighted mean of per-class $F_1$). Accuracy and weighted-$F_1$ are reported too. We pick the best ML model by **test** macro-$F_1$ only (baselines are not eligible). The held-out test set is scored once for the final comparison table.

**Baselines**

1. Always predict `Moderate` (majority class).
2. Social-media hours rule: `Healthy` if `daily_social_media_hours` ≤ 3.5, `At Risk` if > 6.0, else `Moderate`. Cutpoints are fixed round values near train-set class-mean midpoints; they are not tuned on the test set.

Course project only — not for clinical use.


In [ ]:
import evaluation
from io_utils import load_joblib, load_json
from paths import figures_dir, metrics_dir, models_dir
from preprocessing import TRAIN_TEST_SPLIT_FILENAME
import models

diag = features.verify_colab_src_setup()
print("evaluation loaded from:", evaluation.__file__)

# Prefer in-memory Phase 3 splits; fall back to saved artifact.
try:
    X_train_eval = X_train
    X_test_eval = X_test
    y_train_eval = y_train
    y_test_eval = y_test
    print("Using in-memory train/test splits from Phase 3.")
except NameError:
    split_artifact = load_joblib(models_dir() / TRAIN_TEST_SPLIT_FILENAME)
    X_train_eval = split_artifact["X_train_raw"]
    X_test_eval = split_artifact["X_test_raw"]
    y_train_eval = split_artifact["y_train"]
    y_test_eval = split_artifact["y_test"]
    print("Loaded splits from train_test_split.joblib.")

logistic_path = models_dir() / models.LOGISTIC_PIPELINE_FILENAME
rf_path = models_dir() / models.RF_PIPELINE_FILENAME
meta_path = models_dir() / models.PHASE4_MODEL_META_FILENAME
assert logistic_path.is_file(), f"Missing {logistic_path}"
assert rf_path.is_file(), f"Missing {rf_path}"

logistic_eval = load_joblib(logistic_path)
rf_eval = load_joblib(rf_path)
cv_meta = load_json(meta_path) if meta_path.is_file() else {}

print("Train:", len(X_train_eval), "Test:", len(X_test_eval))
print("Train social-media means by class:")
print(
    X_train_eval.assign(**{TARGET: y_train_eval})
    .groupby(TARGET, observed=True)["daily_social_media_hours"]
    .mean()
    .reindex(list(TARGET_CLASSES))
    .round(2)
)
print(
    "Baseline cutpoints (fixed): Healthy <=",
    evaluation.SOCIAL_MEDIA_HEALTHY_MAX,
    "| At Risk >",
    evaluation.SOCIAL_MEDIA_AT_RISK_MIN,
)


In [ ]:
report, comparison_df, report_path, cm_path = evaluation.run_full_evaluation(
    X_train=X_train_eval,
    y_train=y_train_eval,
    X_test=X_test_eval,
    y_test=y_test_eval,
    logistic_pipe=logistic_eval,
    rf_pipe=rf_eval,
    cv_meta=cv_meta,
    figures_dir=figures_dir(),
    metrics_dir=metrics_dir(),
)

print("Best model (test macro-F1):", report["best_model"])
print("Saved report:", report_path)
print("Saved confusion matrix:", cm_path)
print("Overfitting flags:")
for item in report["overfitting_flags"]:
    print(" -", item)

display(comparison_df)


### Confusion matrix (test set)

The figure below is the **test** confusion matrix for the best model by test macro-$F_1$. Rows are true labels, columns are predicted labels; the diagonal counts correct predictions. Saved to `outputs/figures/confusion_matrix_test.png`.


In [ ]:
from IPython.display import Image, display as show_image

show_image(Image(filename=str(cm_path)))
assert report_path.is_file() and report_path.stat().st_size > 0
assert cm_path.is_file() and cm_path.stat().st_size > 0
print("Phase 5 artifacts OK.")


## Evaluation summary and limitations

Both ML models beat the always-Moderate baseline on macro-$F_1$, which matters because Moderate is about 62% of the data. The social-media hours rule is a sanity check only; cutpoints were not optimized on the test set.

The random forest often looks almost perfect on train CV and train scores, so a large train–test gap would mean overfitting rather than a miracle. Prefer the model with the better **test** macro-$F_1$ for the report table.

Limitations to keep in mind: this is a synthetic Kaggle table, not a clinical cohort; predictors are lifestyle-only; At Risk has only 30 test rows so recall for that class is noisy; and none of this is for clinical use. A future sensitivity run could try `class_weight="balanced"` without changing the Phase 4 saved models.


## Export run summary (Part 6)

After a full run, this section writes one ZIP you can download and keep with your project notes:

`outputs/run_summary.zip`

Contents: `run_log.md` (short notes), `manifest.json` (exact numbers), and small CSVs under `tables/`. Edit `RUN_NOTES` below if you want to record anything worth revisiting.


In [ ]:
RUN_NOTES = """
Phase 5 done: scored logistic + RF vs always-Moderate and social-media rule on train/test.
Best model picked by test macro-F1; RF train scores look high so watch the train-test gap.
At Risk test n=30: treat minority recall carefully. Platform crosstab still feels noisy for the report.
"""

from run_summary import write_run_summary
from paths import run_summary_zip_path

zip_path = write_run_summary(
    df=df,
    y_train=y_train,
    y_test=y_test,
    feature_names=feature_names,
    notes=RUN_NOTES.strip(),
)
print("Wrote:", zip_path)
print("Expected path:", run_summary_zip_path())
assert zip_path.is_file() and zip_path.stat().st_size > 0


In [ ]:
try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Not on Colab — zip is at", zip_path)
